<a href="https://colab.research.google.com/github/xzhen-hub/ML_Linear-Factor-Model/blob/main/g1_freddie_julaug_20261102.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Freddie G1: same saved models, June calibration and Julyâ€“August evaluation
Prepared November 2, 2026.

This continues the completed Freddie experiment. It uses **the exact saved expanded Freddie tree** trained May 2021â€“May 2026 and the **October production payoff tree v2-20261002**. Nothing is retrained. Production's underlying tree was pooled across agencies; its separate Freddie served surface/calibration is a different layer. This comparison uses the direct payoff trees, not the complete deployed surface or voluntary CPR.

The question is whether the historical G1 improvement continues in July and August, especially for delinquent and small loans. June is calibration-only. July and August never select trees, features or calibration factors. These are later-calendar research tests, not a newly untouched blind test or proof that the split alone caused improvements.

Use the same research scope: verified 2024 adjusted-vintage Freddie 30-year pools, coupon buckets 6.0 and 6.5. It is not certified as the full official DPR G1 membership. Reconstruct membership separately each month; do not require survival into August. All available matched loans are evaluated; there is no evaluation negative sampling.

## Before starting
1. Open this `.ipynb` using Colab **File â†’ Upload notebook**. Save a copy in Google Drive. A new notebook in the same Colab browser is fine; it has its own setup and does not depend on old session variables.
2. Use a CPU runtime. Upload the following file using the **Files sidebar â†’ Upload**:
   `g1_freddie_full61_20261102_20261102T065244_c17762d1_FULL_recovery.zip`
   Local location: `Y:/OASIVE/`. The size is **656,849,168 bytes**. Wait until the upload finishes. The small-results ZIP is insufficient. Do not upload either Fannie ZIP.
3. Run Steps 1â€“15 in order. If Step 2 installs packages, restart the session once, then rerun Steps 1â€“2 and continue.

All persistent BigQuery writes go to unique tables in `gen-lang-client-0343560978.oasive_quant_sandbox`, with 30-day expiry. Source tables and the production model are read-only. No bucket is needed. Local `/content` files disappear when the runtime expires; download the backups at Step 15.

Three separately reported stages:
- `before`: both raw tree odds receive the same 0.12 negative-sampling correction, once.
- `june_separate`: each model gets its own additional June-only odds adjustment, then those adjustments are frozen.
- `june_common_production`: production's June adjustment is applied to both as a diagnostic.

The registered Freddie raw factor 0.1185 is not applied on top of 0.12. June adjustments here are research calibrations on this G1 cohort, not replicas of deployed production calibration.


## Step 1 â€” Restore exact assets from the Freddie FULL backup
Run after the upload finishes. Change `INPUT_FULL` only if Colab renamed the file. A truncated ZIP must be uploaded again. The recovery verifies hashes and rejects conflicting files.

To resume after a later runtime loss, upload this notebook's own latest FULL recovery ZIP and set `RESTORE_FOLLOWUP_ZIP` below. Keep the same root. You do not need the old training data or the old BigQuery evaluation panel.


In [1]:
from pathlib import Path
import json,hashlib,zipfile,shutil,sys,subprocess,importlib,importlib.util,gc,uuid
ROOT=Path('/content/g1_freddie_julaug_20261102');ROOT.mkdir(exist_ok=True)
INPUTS=ROOT/'inputs'
INPUT_FULL=Path('/content/g1_freddie_full61_20261102_20261102T065244_c17762d1_FULL_recovery.zip')
RESTORE_FOLLOWUP_ZIP=None  # Resume: '/content/<this-notebooks-newest-FULL_recovery.zip>'

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024**2),b''):h.update(b)
    return h.hexdigest()

def restore(archive,dest,manifest_name,selected=None):
    archive=Path(archive);dest=Path(dest)
    assert archive.is_file() and zipfile.is_zipfile(archive),f'Missing/incomplete ZIP: {archive}'
    with zipfile.ZipFile(archive) as z:
        hashes=json.loads(z.read(manifest_name));names=list(hashes) if selected is None else selected
        for name in names:
            assert name in hashes,'Wrong ZIP; missing '+name
            p=(dest/name).resolve();assert p.is_relative_to(dest.resolve()),'Unsafe ZIP path'
            if not p.exists():
                p.parent.mkdir(parents=True,exist_ok=True);temp=p.with_suffix(p.suffix+'.recovering')
                with z.open(name) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst,1024**2)
                assert sha(temp)==hashes[name],'ZIP checksum mismatch: '+name
                temp.replace(p)
            assert sha(p)==hashes[name],'Existing file conflicts with backup: '+name
        return {n:hashes[n] for n in names}
selected=['evaluation/roster_table_identity.json','evaluation/run_spec.json','evaluation/preparation_spec.json','evaluation/completion.json',
 'evaluation/months/2026-03-01/paired_predictions.parquet',
 'tools/g1_saved_payoff_comparison_core.py','tools/freddie_research_pool_roster.csv',
 'expanded_freddie_25features_seed20261021/freddie_expanded_25features.ubj',
 'expanded_freddie_25features_seed20261021/fixed_freddie_encoder.joblib',
 'expanded_freddie_25features_seed20261021/experiment_spec.json',
 'expanded_freddie_25features_seed20261021/fit_complete.json',
 'original/original_saved_experiment/post_rally_matched_test.py',
 'original/original_saved_experiment/manifest.json']
if RESTORE_FOLLOWUP_ZIP:
    restore(RESTORE_FOLLOWUP_ZIP,ROOT,'freddie_julaug_recovery_hashes.json')
elif not (INPUTS/'selected_input_hashes.json').exists():
    h=restore(INPUT_FULL,INPUTS,'freddie_recovery_hashes.json',selected)
    (INPUTS/'selected_input_hashes.json').write_text(json.dumps(h,indent=2))
for name,h in json.loads((INPUTS/'selected_input_hashes.json').read_text()).items():assert sha(INPUTS/name)==h
OLD_SPEC=json.loads((INPUTS/'evaluation/run_spec.json').read_text())
OLD_PREP=json.loads((INPUTS/'evaluation/preparation_spec.json').read_text())
MANIFEST=json.loads((INPUTS/'original/original_saved_experiment/manifest.json').read_text())
assert OLD_SPEC['training_start']=='2021-05-01' and OLD_SPEC['training_last_month']=='2026-05-01'
assert OLD_SPEC['asset_sha256']['model']=='8489bf8766894b0c5c46a1db65d6567c2cb6357de563c5571b153c2d0f3264b8'
print('Verified Freddie inputs:',INPUTS)


Verified Freddie inputs: /content/g1_freddie_julaug_20261102/inputs


## Step 2 â€” Match the recorded software versions
This may install packages. If it asks for a restart, select **Runtime â†’ Restart session**, then rerun Steps 1â€“2. Do not retrain or replace the encoder to solve a package mismatch.


In [2]:
import importlib.metadata
requirements=[]
for name,version in MANIFEST['versions'].items():
    try:current=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:current=None
    if current!=version:requirements.append(name+'=='+version)
if requirements:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*requirements],check=True)
    raise RuntimeError('Packages installed. Restart session, rerun Steps 1â€“2, then continue.')
print('Exact recorded package versions match.')


Exact recorded package versions match.


## Step 3 â€” Load the follow-up helper
This helper is embedded in the notebook. No separate code upload is required. Do not run old repair cells in this notebook.


In [3]:
%%writefile /content/g1_freddie_julaug.py
"""Freddie saved-tree June calibration and July-August evaluation. No training."""
from pathlib import Path
import json,hashlib,re
import numpy as np,pandas as pd
import g1_saved_payoff_comparison_core as core
PROJECT=core.PROJECT
MODEL=core.MODEL
FEATURES=PROJECT+'.oasive_quant_bq.feature_loan_month_mat_bak20261005_171029'
LABELS=PROJECT+'.oasive_quant_bq.label_loan_month_fast_bak20261010_193027'
FACTS=PROJECT+'.oasive_quant_bq.fact_loan_monthly_sftp'
DIM=PROJECT+'.oasive_quant_bq.dim_loan'
SOURCES=[FEATURES,LABELS,FACTS,DIM]
CAL_MONTH='2026-06-01'
EVAL_MONTHS=['2026-07-01','2026-08-01']
MONTHS=[CAL_MONTH]+EVAL_MONTHS
STAGES=['before','june_separate','june_common_production']
CAP=500*1024**3
READ_CAP=10*1024**3
TOTAL_CAP=1000*1024**3

def sandbox_table(name):
 if not re.fullmatch(re.escape(PROJECT)+r'\.oasive_quant_sandbox\.g1_freddie_julaug_[a-z0-9_]+',name):raise ValueError('Unique Freddie follow-up sandbox tables only')
 return name

def members_cte(roster):
 sandbox_table(roster)
 return f"""WITH target AS (SELECT * FROM `{roster}` WHERE mapping_status='known_target'),
 members AS (
 SELECT 'freddie' agency,CAST(a.loan_id AS STRING) loan_id,a.as_of_date feature_date,a.pool_id,
 p.pool_coupon,p.actual_coupon,p.issue_date,p.pool_vintage,
 CAST(a.current_upb AS FLOAT64) current_upb
 FROM `{FACTS}` a JOIN `{DIM}` d ON a.loan_id=d.loan_id
 JOIN target p ON a.pool_id=p.pool_id
 WHERE a.as_of_date BETWEEN DATE '2026-06-01' AND DATE '2026-08-01'
 AND a.as_of_date=DATE_TRUNC(a.as_of_date,MONTH) AND d.orig_term=360
 AND a.current_upb>0 AND p.issue_date<=a.as_of_date)"""

def census_sql(roster):
 return members_cte(roster)+""", counts AS (
 SELECT feature_date,pool_coupon,COUNT(*) member_rows,COUNT(DISTINCT loan_id) distinct_member_loans,SUM(current_upb) member_upb
 FROM members GROUP BY 1,2)
 SELECT month feature_date,coupon pool_coupon,
 (SELECT COUNT(*) FROM target p WHERE p.pool_coupon=coupon AND p.issue_date<=month) issued_target_pools,
 COALESCE(c.member_rows,0) member_rows,COALESCE(c.distinct_member_loans,0) distinct_member_loans,COALESCE(c.member_upb,0) member_upb
 FROM UNNEST(GENERATE_DATE_ARRAY(DATE '2026-06-01',DATE '2026-08-01',INTERVAL 1 MONTH)) month
 CROSS JOIN UNNEST([6.0,6.5]) coupon LEFT JOIN counts c ON c.feature_date=month AND c.pool_coupon=coupon ORDER BY 1,2"""

def freeze_sql(roster,panel,feature_start):
 sandbox_table(panel)
 if feature_start not in MONTHS:raise ValueError('Invalid static partition start')
 expressions=[]
 for c in core.BASE:
  e='f.'+c
  if c in ['occupancy','channel','property_type']:e=f"COALESCE(f.{c},'UNK')"
  if c=='num_units':e='COALESCE(f.num_units,1)'
  expressions.append(e+' AS '+c)
 return f"""CREATE TABLE `{panel}` PARTITION BY feature_date CLUSTER BY pool_id,loan_id
 OPTIONS(expiration_timestamp=TIMESTAMP_ADD(CURRENT_TIMESTAMP(),INTERVAL 30 DAY),description='Freddie G1 June-August saved-tree comparison; research only') AS
 """+members_cte(roster)+f"""
 SELECT m.*,f.loan_id IS NOT NULL feature_record_present,CAST(f.current_upb AS FLOAT64) feature_upb,
 {', '.join(expressions)},l.loan_id IS NOT NULL label_record_present,
 l.is_curtailment IS NOT NULL training_label_eligible,l.termination_component,
 IF(l.loan_id IS NULL,NULL,IF(l.termination_component IN('refinance','turnover','cashout','buyout'),1,0)) actual_payoff,
 IF(l.loan_id IS NULL,NULL,IF(l.termination_component IN('refinance','turnover','cashout','buyout','turnover_proxy','voluntary_unknown'),1,0)) broad_payoff
 FROM members m LEFT JOIN `{FEATURES}` f ON f.agency=m.agency AND f.loan_id=m.loan_id AND f.as_of_date=m.feature_date
 AND f.as_of_date BETWEEN DATE '{feature_start}' AND DATE '2026-08-01' AND f.orig_term=360
 LEFT JOIN `{LABELS}` l ON l.agency=m.agency AND l.loan_id=m.loan_id AND l.as_of_date=m.feature_date
 AND l.as_of_date BETWEEN DATE '2026-06-01' AND DATE '2026-08-01'"""

def panel_audit_sql(panel):
 return f"""SELECT feature_date,pool_coupon,COUNT(*) panel_rows,COUNT(DISTINCT loan_id) distinct_panel_loans,
 SUM(current_upb) panel_upb,SUM(actual_payoff) payoff_events,
 COUNTIF(feature_record_present IS NOT TRUE) missing_features,
 COUNTIF(label_record_present IS NOT TRUE OR actual_payoff IS NULL) missing_labels,
 COUNTIF(training_label_eligible IS NOT TRUE) training_rule_ineligible_rows,
 COUNTIF(actual_payoff NOT IN (0,1) OR actual_payoff IS DISTINCT FROM broad_payoff) label_definition_gaps,
 COUNTIF(feature_upb IS NULL OR ABS(current_upb-feature_upb)>.02) balance_conflicts
 FROM `{panel}` GROUP BY 1,2 ORDER BY 1,2"""

def month_sql(panel,month):
 if month not in MONTHS:raise ValueError('Outside June-August calendar')
 where=f"agency='freddie' AND feature_date=DATE '{month}'"
 cols=', '.join(core.KEY+['pool_id','pool_coupon','current_upb']+core.BASE)
 return f'SELECT * FROM `{panel}` WHERE {where}',f"""SELECT agency,loan_id,feature_date,pool_id,pool_coupon,current_upb,
 (SELECT x.prob FROM UNNEST(predicted_is_payoff_probs) x WHERE SAFE_CAST(x.label AS INT64)=1) production_raw
 FROM ML.PREDICT(MODEL `{MODEL}`,(SELECT {cols} FROM `{panel}` WHERE {where}))"""

def cached_query(client,name,query,root,limit=CAP):
 from google.cloud import bigquery
 root=Path(root);folder=root/'query_cache';folder.mkdir(exist_ok=True)
 path=folder/(name+'.parquet');done=folder/(name+'_complete.json');started=folder/(name+'_job.json');digest=hashlib.sha256(query.encode()).hexdigest()
 if not query.lstrip().upper().startswith(('SELECT','WITH')):raise ValueError('Read cache accepts SELECT/WITH only')
 if done.exists():
  d=json.loads(done.read_text())
  if d['sql_sha256']!=digest or core.sha(path)!=d['file_sha256']:raise ValueError('Changed cached query')
  return pd.read_parquet(path)
 if started.exists():
  d=json.loads(started.read_text())
  if d['sql_sha256']!=digest:raise ValueError('Changed query')
  job=client.get_job(d['job_id'],location='us-central1')
 else:
  job=client.query(query,job_config=bigquery.QueryJobConfig(maximum_bytes_billed=limit),location='us-central1')
  core.write_json(started,dict(job_id=job.job_id,sql_sha256=digest))
 print(name,'job:',job.job_id,flush=True)
 pieces=[pd.DataFrame([dict(x) for x in page]) for page in job.result(page_size=10000).pages]
 frame=pd.concat(pieces,ignore_index=True) if pieces else pd.DataFrame();frame.to_parquet(path,index=False)
 core.write_json(done,dict(job_id=job.job_id,sql_sha256=digest,file_sha256=core.sha(path),rows=len(frame)));return frame

def assert_run(client,root,spec):
 root=Path(root)
 if core.sha(__file__)!=spec['helper_sha256'] or core.sha(core.__file__)!=spec['core_sha256']:raise ValueError('Comparison code changed')
 if core.table_id(client.get_table(spec['panel']))!=spec['panel_identity']:raise ValueError('Frozen panel changed or expired')
 if core.model_id(client.get_model(MODEL))!=spec['production_identity']:raise ValueError('Original October model changed')
 for name,path in spec['asset_paths'].items():
  if core.sha(path)!=spec['asset_sha256'][name]:raise ValueError('Candidate/matrix asset changed: '+name)

def validate_inventory(c,a):
 c=c.copy();a=a.copy()
 for d in [c,a]:d['feature_date']=pd.to_datetime(d.feature_date).dt.strftime('%Y-%m-%d')
 expected={(mo,cp) for mo in MONTHS for cp in [6.,6.5]}
 for name,d in [('census',c),('panel',a)]:
  if d.duplicated(['feature_date','pool_coupon']).any() or set(zip(d.feature_date,d.pool_coupon))!=expected:raise ValueError(name+' must have all six June-August coupon cells')
 if not c.member_rows.gt(0).all() or not c.member_rows.eq(c.distinct_member_loans).all():raise ValueError('Empty or duplicate membership: reconcile before scoring')
 z=c.merge(a,on=['feature_date','pool_coupon'],validate='one_to_one')
 if not z.member_rows.eq(z.panel_rows).all() or not z.panel_rows.eq(z.distinct_panel_loans).all():raise ValueError('Feature/label join lost or duplicated loans')
 if not np.allclose(z.member_upb.to_numpy(float),z.panel_upb.to_numpy(float),atol=.1,rtol=1e-12):raise ValueError('Panel/census UPB differs')
 if not z[['missing_features','missing_labels','label_definition_gaps','balance_conflicts']].eq(0).all().all():raise ValueError('Resolve genuine feature/label/definition/balance errors')
 if (z.member_rows.ge(10000)&z.payoff_events.eq(0)).any():raise ValueError('Large zero-payoff cell: investigate outcome completeness; do not interpret as actual zero')
 return z

def score_month(client,root,spec,month,inventory,model,encoder,m):
 root=Path(root);assert_run(client,root,spec)
 folder=root/'months'/month;folder.mkdir(parents=True,exist_ok=True)
 marker=folder/'scoring_complete.json';path=folder/'paired_predictions_before.parquet'
 if marker.exists():
  d=json.loads(marker.read_text())
  if d['spec_sha256']!=core.sha(root/'run_spec.json'):raise ValueError('Changed scoring lock')
  for n,h in d['files'].items():
   if core.sha(folder/n)!=h:raise ValueError('Changed prediction/category file')
  return pd.read_parquet(path)
 iq,pq=month_sql(spec['panel'],month)
 f=core.normalize(cached_query(client,'input_'+month,iq,root,READ_CAP))
 if not f.agency.eq('freddie').all() or not f.feature_date.eq(month).all():raise ValueError('Wrong month/agency')
 if not f.feature_record_present.eq(True).all() or not f.label_record_present.eq(True).all():raise ValueError('Missing feature/label record')
 if not f.actual_payoff.isin([0,1]).all() or not f.actual_payoff.eq(f.broad_payoff).all():raise ValueError('Invalid payoff definition')
 inv=inventory.loc[inventory.feature_date.eq(month)]
 if len(f)!=int(inv.member_rows.sum()):raise ValueError('Monthly count mismatch')
 for row in inv.itertuples():
  g=f.loc[f.pool_coupon.eq(row.pool_coupon)]
  if len(g)!=int(row.member_rows) or int(g.actual_payoff.sum())!=int(row.payoff_events) or not np.isclose(g.current_upb.sum(),row.member_upb,atol=.1,rtol=1e-12):raise ValueError('Downloaded coupon differs from audit')
 pred=core.normalize(cached_query(client,'predict_'+month,pq,root,READ_CAP));raw=pred.pop('production_raw').to_numpy(float)
 if not np.isfinite(raw).all() or ((raw<0)|(raw>1)).any():raise ValueError('Invalid production probability')
 pred['production_before']=odds(raw,.12)
 paired=core.join_predictions(f,pred,core.score_expanded(model,encoder,m,f));paired.to_parquet(path,index=False)
 cats=m.cats(f);rows=[]
 for i,col in enumerate(core.CAT):
  unknown=~cats[col].isin(encoder.categories_[i]);rows.append(dict(month=month,feature=col,unknown_rows=int(unknown.sum()),unknown_upb_share=float(f.loc[unknown,'current_upb'].sum()/f.current_upb.sum())))
 pd.DataFrame(rows).to_csv(folder/'category_coverage.csv',index=False)
 core.write_json(marker,dict(spec_sha256=core.sha(root/'run_spec.json'),files={p.name:core.sha(p) for p in [path,folder/'category_coverage.csv']}))
 assert_run(client,root,spec);return paired

def odds(p,factor):
 p=np.clip(np.asarray(p,dtype=float),1e-12,1-1e-12)
 if not np.isfinite(factor) or factor<=0:raise ValueError('Invalid odds factor')
 return factor*p/(1-p+factor*p)

def june_calibration(june,module):
 june=core.normalize(june)
 if not june.feature_date.eq(CAL_MONTH).all():raise ValueError('Calibration must use June only; July-August outcomes forbidden')
 y=june.actual_payoff.to_numpy(float);w=june.current_upb.to_numpy(float)
 if not set(np.unique(y))=={0,1} or not np.isfinite(w).all() or (w<=0).any():raise ValueError('Invalid calibration outcomes or weights')
 factors={};rows=[]
 for arm in ['production','expanded']:
  p=june[arm+'_before'].to_numpy(float)
  factor=module.solve_factor(p,y,w);factors[arm]=factor
  rows.append(dict(month=CAL_MONTH,agency='freddie',arm=arm,scope='all selected G1 coupons combined',loans=len(june),payoff_events=int(y.sum()),upb=float(w.sum()),actual_pct=100*float(np.average(y,weights=w)),before_pct=100*float(np.average(p,weights=w)),after_pct=100*float(np.average(odds(p,factor),weights=w)),factor_on_before_probability=factor,equivalent_factor_on_raw_probability=.12*factor))
 return factors,pd.DataFrame(rows)

def report(f,month,stage):
 if month not in EVAL_MONTHS:raise ValueError('June must not enter evaluation results')
 out=core.group_report(f,month);extra=[]
 # Whole-book small balances, plus balance by payment status: not restricted to young/high-incentive loans.
 for coupon in [6.,6.5]:
  cp=f.pool_coupon.eq(coupon);size=pd.to_numeric(f.orig_upb);dlq=pd.to_numeric(f.dlq_days)
  groups=[('small_all_ages',str(coupon)+'_lt150k',cp&size.lt(150000))]
  for label,mask in [('lt150k',size.lt(150000)),('150_300k',size.ge(150000)&size.lt(300000)),('300_500k',size.ge(300000)&size.lt(500000)),('500kplus',size.ge(500000))]:
   for status,s in [('current',dlq.eq(0)),('delinquent',dlq.gt(0)),('unknown',dlq.isna()|dlq.lt(0))]:groups.append(('status_balance',f'{coupon}_{status}_{label}',cp&mask&s))
  for key,name,mask in groups:
   if mask.any():extra.append(dict(month=month,grouping=key,group=name,**core.metric(f.loc[mask])))
 out=pd.concat([out,pd.DataFrame(extra)],ignore_index=True)
 out['stage']=stage;out['agency']='freddie';out['role']='evaluation'
 return out

def report_stages(f,month,factors):
 if month not in EVAL_MONTHS:raise ValueError('Evaluation is July-August only')
 frames=[]
 for stage in STAGES:
  x=f.copy()
  if stage!='before':
   for arm in ['production','expanded']:
    factor=factors[arm] if stage=='june_separate' else factors['production']
    x[arm+'_before']=odds(f[arm+'_before'],factor)
  frames.append(report(x,month,stage))
 return pd.concat(frames,ignore_index=True)

def summarize(results,maturity,june_review=None):
 if not set(results.month).issubset(EVAL_MONTHS):raise ValueError('Evaluation summary includes non-evaluation month')
 if maturity.duplicated('month').any() or set(maturity.month)!=set(EVAL_MONTHS):raise ValueError('Require one maturity review row per evaluation month')
 if not maturity.status.isin(['unconfirmed','provisional','confirmed']).all():raise ValueError('Invalid maturity status')
 if maturity.loc[maturity.status.eq('confirmed'),'evidence'].fillna('').str.strip().eq('').any():raise ValueError('Confirmed outcome status requires recorded source-completeness evidence')
 review=june_review or {'status':'unconfirmed','evidence':''}
 if review['status'] not in ['unconfirmed','provisional','confirmed']:raise ValueError('Invalid June maturity status')
 if review['status']=='confirmed' and not review.get('evidence','').strip():raise ValueError('Confirmed June outcomes require evidence')
 results=results.merge(maturity.rename(columns={'status':'outcome_status','evidence':'outcome_evidence'}),on='month',validate='many_to_one')
 results['calibration_outcome_status']=np.where(results.stage.eq('before'),'not_applicable',review['status'])
 rows=[]
 for name,months in [('jul_aug',EVAL_MONTHS)]:
  for key,g in results.loc[results.month.isin(months)].groupby(['stage','grouping','group'],observed=True):
   w=g.upb.to_numpy(float);avg=lambda c:float(np.average(g[c],weights=w))
   d=dict(period=name,stage=key[0],grouping=key[1],group=key[2],months=g.month.nunique(),expected_months=len(months),loan_months=int(g.loans.sum()),upb_month_exposure_bn=float(w.sum()/1e9),all_outcomes_confirmed=bool(g.outcome_status.eq('confirmed').all()))
   d['calibration_outcomes_confirmed']=key[0]=='before' or review['status']=='confirmed'
   d['complete_group_month_coverage']=d['months']==d['expected_months']
   d['ready_for_final_interpretation']=d['all_outcomes_confirmed'] and d['calibration_outcomes_confirmed'] and d['complete_group_month_coverage']
   for arm in ['production','expanded']:
    d[arm+'_weighted_abs_bias_pp']=float(np.average(abs(g[arm+'_bias_pp']),weights=w));d[arm+'_weighted_signed_bias_pp']=avg(arm+'_bias_pp')
   d['weighted_abs_bias_gain_pp']=d['production_weighted_abs_bias_pp']-d['expanded_weighted_abs_bias_pp']
   for loss in ['brier','logloss']:d['weighted_'+loss+'_gain']=avg(loss+'_gain')
   rows.append(d)
 return results,pd.DataFrame(rows)

def coupon_no_offset_summary(results):
 rows=[]
 for stage,g in results.loc[results.grouping.eq('coupon')].groupby('stage'):
  if set(zip(g.month,g['group']))!={(mo,str(cp)) for mo in EVAL_MONTHS for cp in [6.,6.5]}:raise ValueError('Need four coupon-month cells')
  w=g.upb.to_numpy(float);d=dict(stage=stage,coupon_month_cells=len(g),upb_month_exposure_bn=float(w.sum()/1e9))
  for arm in ['production','expanded']:d[arm+'_weighted_abs_bias_pp']=float(np.average(abs(g[arm+'_bias_pp']),weights=w))
  d['weighted_abs_bias_gain_pp']=d['production_weighted_abs_bias_pp']-d['expanded_weighted_abs_bias_pp']
  for loss in ['brier','logloss']:d['weighted_'+loss+'_gain']=float(np.average(g[loss+'_gain'],weights=w))
  rows.append(d)
 return pd.DataFrame(rows)


Writing /content/g1_freddie_julaug.py


## Step 4 â€” Load exact saved Freddie model and replay March
The replay checks that the restored model, encoder and matrix reproduce the completed experiment. March is not used to choose July/August settings. Expected March rows: **259,110**.


In [4]:
import numpy as np,pandas as pd,joblib,xgboost as xgb

def load_exact(name,path):
    s=importlib.util.spec_from_file_location(name,str(path));mod=importlib.util.module_from_spec(s)
    sys.modules[name]=mod;s.loader.exec_module(mod);return mod
core=load_exact('g1_saved_payoff_comparison_core',INPUTS/'tools/g1_saved_payoff_comparison_core.py')
follow=load_exact('g1_freddie_julaug',Path('/content/g1_freddie_julaug.py'))
assert sha(core.__file__)==OLD_SPEC['core_sha256']
EP=INPUTS/'expanded_freddie_25features_seed20261021'
ASSETS={'model':str(EP/'freddie_expanded_25features.ubj'),
        'encoder':str(EP/'fixed_freddie_encoder.joblib'),
        'module':str(INPUTS/'original/original_saved_experiment/post_rally_matched_test.py')}
for role,p in ASSETS.items():assert sha(p)==OLD_SPEC['asset_sha256'][role],role+' asset mismatch'
m=load_exact('freddie_julaug_saved_matrix',ASSETS['module'])
assert m.BASE==core.BASE and m.CAT==core.CAT and m.Q==.12
ENCODER=joblib.load(ASSETS['encoder'])
EXPANDED=xgb.Booster(model_file=ASSETS['model']);EXPANDED.set_param({'nthread':2,'device':'cpu'})
golden=core.normalize(pd.read_parquet(INPUTS/'evaluation/months/2026-03-01/paired_predictions.parquet'))
assert len(golden)==259110 and golden.agency.eq('freddie').all()
replay=core.score_expanded(EXPANDED,ENCODER,m,golden)
assert np.allclose(replay,golden.expanded_before,atol=1e-7,rtol=0),'Saved-model replay mismatch'
core.write_json(ROOT/'saved_model_replay.json',dict(loans=len(golden),max_error=float(np.max(abs(replay-golden.expanded_before)))))
CODE=ROOT/'code';CODE.mkdir(exist_ok=True);shutil.copy2(follow.__file__,CODE/'g1_freddie_julaug.py')
del golden,replay;gc.collect()
print('Same Freddie expanded tree verified. No fitting performed.')


Same Freddie expanded tree verified. No fitting performed.


## Step 5 â€” Authenticate, pin sources, and restore the pool roster into your sandbox
Use your existing Google account. The archived roster is restored to a new sandbox table so this test can resume even after the old panel/roster expires. It must contain exactly 12,130 known-target pools. The table is created once with WRITE_EMPTY, never overwritten, and expires after 30 days.

Feature and label snapshots must match the old experiment. Changes to servicing/dimension sources are recorded; those sources must then stay fixed while the panel is prepared.


In [5]:
from google.colab import auth
from google.cloud import bigquery
from google.api_core.exceptions import NotFound
from datetime import datetime,timedelta,timezone
auth.authenticate_user()
CLIENT=bigquery.Client(project=core.PROJECT,location='us-central1')
PROD_ID=core.model_id(CLIENT.get_model(core.MODEL))
assert PROD_ID==OLD_SPEC['production_identity'],'October production artifact changed'
prep_path=ROOT/'preparation_spec.json'
if prep_path.exists():
    PREP=json.loads(prep_path.read_text())
    assert PREP['helper_sha256']==sha(follow.__file__) and PREP['core_sha256']==sha(core.__file__)
    assert PREP['production_identity']==PROD_ID and PREP['months']==follow.MONTHS
else:
    pinned={s:core.table_id(CLIENT.get_table(s)) for s in follow.SOURCES}
    for s in [follow.FEATURES,follow.LABELS]:assert pinned[s]==OLD_PREP['source_identities'][s],'Historical snapshot changed: '+s
    for s in follow.SOURCES:assert CLIENT.get_table(s).location.lower()=='us-central1'
    stem=core.PROJECT+'.oasive_quant_sandbox.g1_freddie_julaug_'+uuid.uuid4().hex[:12]
    PREP=dict(months=follow.MONTHS,panel=stem+'_panel',roster=stem+'_roster',source_identities=pinned,
              production_identity=PROD_ID,helper_sha256=sha(follow.__file__),core_sha256=sha(core.__file__),
              roster_sha256=sha(INPUTS/'tools/freddie_research_pool_roster.csv'))
    core.write_json(prep_path,PREP)
roster_path=INPUTS/'tools/freddie_research_pool_roster.csv'
assert sha(roster_path)==PREP['roster_sha256']==OLD_PREP['roster_csv_sha256']
roster=pd.read_csv(roster_path,dtype={'pool_id':str})
assert not roster.pool_id.duplicated().any()
assert int(roster.mapping_status.eq('known_target').sum())==12130
upload=roster[['pool_id','pool_coupon','actual_coupon','issue_date','pool_vintage','mapping_status']].copy()
upload['issue_date']=pd.to_datetime(upload.issue_date).dt.date
upload['pool_vintage']=pd.to_numeric(upload.pool_vintage).astype('Int64')
assert roster.agency.eq('freddie').all()
# Use the archived table schema, rather than infer identifier/date types.
schema=[bigquery.SchemaField.from_api_repr(x) for x in json.loads((INPUTS/'evaluation/roster_table_identity.json').read_text())['schema']['fields']]
try:t=CLIENT.get_table(PREP['roster'])
except NotFound:
    t=bigquery.Table(PREP['roster'],schema=schema);t.expires=datetime.now(timezone.utc)+timedelta(days=30)
    t.description='Exact archived Freddie research pool roster; saved-tree July-August follow-up'
    t=CLIENT.create_table(t)
roster_lock=ROOT/'roster_identity.json'
if roster_lock.exists():assert core.table_id(t)==json.loads(roster_lock.read_text()),'Restored roster changed'
else:
    marker=ROOT/'roster_load_job.json'
    if marker.exists():
        d=json.loads(marker.read_text());assert d['csv_sha256']==PREP['roster_sha256']
        job=CLIENT.get_job(d['job_id'],location='us-central1')
    else:
        cfg=bigquery.LoadJobConfig(schema=schema,write_disposition=bigquery.WriteDisposition.WRITE_EMPTY)
        job=CLIENT.load_table_from_dataframe(upload,PREP['roster'],job_config=cfg,location='us-central1')
        core.write_json(marker,dict(job_id=job.job_id,csv_sha256=PREP['roster_sha256']))
    job.result();t=CLIENT.get_table(PREP['roster']);assert int(t.num_rows)==len(roster)
    core.write_json(roster_lock,core.table_id(t))
if not (ROOT/'frozen_panel_identity.json').exists():
    for s,identity in PREP['source_identities'].items():assert core.table_id(CLIENT.get_table(s))==identity,'Source changed: '+s
print('Sandbox roster:',PREP['roster'])
print('New panel:',PREP['panel'])


Sandbox roster: gen-lang-client-0343560978.oasive_quant_sandbox.g1_freddie_julaug_74f83b69ecb9_roster
New panel: gen-lang-client-0343560978.oasive_quant_sandbox.g1_freddie_julaug_74f83b69ecb9_panel


## Step 6 â€” Dry-run the preparation queries
Dry runs estimate scan volume without executing the census or panel creation. Review estimates before Step 7. Development dry runs using the existing archived Freddie roster estimated approximately **28.05 GiB for the census** and **105.29 GiB for the panel freeze**; your fresh dry runs are authoritative. Per preparation query cap: 500 GiB; total planned cap including scoring: 1,000 GiB. These are caps, not quoted bills. Cached downloads avoid repeating completed work.


In [6]:
def dry(q,cap):
    j=CLIENT.query(q,job_config=bigquery.QueryJobConfig(dry_run=True,use_query_cache=False,maximum_bytes_billed=cap),location='us-central1')
    n=int(j.total_bytes_processed);assert n<=cap,'Query exceeds declared scan cap';return n
SQL=ROOT/'sql';SQL.mkdir(exist_ok=True)
QUERIES={'membership_census':follow.census_sql(PREP['roster']),
         'freeze_panel':follow.freeze_sql(PREP['roster'],PREP['panel'],'2026-06-01')}
rows=[]
for name,q in QUERIES.items():
    (SQL/(name+'.sql')).write_text(q)
    n=dry(q,follow.CAP);rows.append(dict(query=name,estimated_bytes=n,estimated_GiB=n/1024**3))
EST=pd.DataFrame(rows);assert EST.estimated_bytes.sum()<=follow.TOTAL_CAP
EST.to_csv(ROOT/'preparation_estimates.csv',index=False)
print(EST.to_string(index=False))


            query  estimated_bytes  estimated_GiB
membership_census      30117072994      28.048710
     freeze_panel     113055181137     105.290842


## Step 7 â€” Run the independent membership census
Expect six positive cells: June, July and August Ã— two coupons. Counts can change each month. Requiring a February row, March membership or survival into August would change the test and is not allowed.


In [7]:
CENSUS=follow.cached_query(CLIENT,'membership_census',QUERIES['membership_census'],ROOT)
CENSUS['feature_date']=pd.to_datetime(CENSUS.feature_date).dt.strftime('%Y-%m-%d')
assert set(zip(CENSUS.feature_date,CENSUS.pool_coupon))=={(mo,cp) for mo in follow.MONTHS for cp in [6.,6.5]}
assert not CENSUS.duplicated(['feature_date','pool_coupon']).any()
assert CENSUS.member_rows.gt(0).all() and CENSUS.member_rows.eq(CENSUS.distinct_member_loans).all()
CENSUS.to_csv(ROOT/'membership_census.csv',index=False)
print(CENSUS.to_string(index=False))


membership_census job: 8369e970-55fc-4052-a35a-1c692f2ffb22
feature_date  pool_coupon  issued_target_pools  member_rows  distinct_member_loans   member_upb
  2026-06-01          6.0                 7163       158754                 158754 4.677318e+10
  2026-06-01          6.5                 4967        83058                  83058 2.331798e+10
  2026-07-01          6.0                 7163       156657                 156657 4.605365e+10
  2026-07-01          6.5                 4967        81431                  81431 2.278249e+10
  2026-08-01          6.0                 7163       154668                 154668 4.537949e+10
  2026-08-01          6.5                 4967        79867                  79867 2.227030e+10


## Step 8 â€” Freeze the matched panel in the sandbox
This executes the large preparation query once. It includes the same 25 model features, current-UPB weights and payoff definition as the completed Freddie test. Null curtailment training eligibility is recorded but does not remove valid payoff rows.


In [8]:
identity_path=ROOT/'frozen_panel_identity.json'
if identity_path.exists():
    PANEL_ID=json.loads(identity_path.read_text())
    assert core.table_id(CLIENT.get_table(PREP['panel']))==PANEL_ID,'Panel changed/expired: use saved predictions or a new run'
else:
    for s,identity in PREP['source_identities'].items():assert core.table_id(CLIENT.get_table(s))==identity,'Source changed: '+s
    assert core.table_id(CLIENT.get_table(PREP['roster']))==json.loads(roster_lock.read_text())
    marker=ROOT/'freeze_job.json';q=QUERIES['freeze_panel'];digest=hashlib.sha256(q.encode()).hexdigest()
    if marker.exists():
        d=json.loads(marker.read_text());assert d['sql_sha256']==digest
        job=CLIENT.get_job(d['job_id'],location='us-central1')
    else:
        job=CLIENT.query(q,job_config=bigquery.QueryJobConfig(maximum_bytes_billed=follow.CAP),location='us-central1')
        core.write_json(marker,dict(job_id=job.job_id,sql_sha256=digest))
    print('Freeze job:',job.job_id);job.result()
    for s,identity in PREP['source_identities'].items():assert core.table_id(CLIENT.get_table(s))==identity,'Source changed during freeze'
    PANEL_ID=core.table_id(CLIENT.get_table(PREP['panel']));core.write_json(identity_path,PANEL_ID)
print('Frozen panel rows:',PANEL_ID['numRows'])


Freeze job: 83e3c378-b5cc-46a1-9294-3d1516116975
Frozen panel rows: 714435


## Step 9 â€” Validate the matched data and inspect payoff coverage
All six cells must match independent membership counts and balances. Missing features/labels, duplicate rows, balance conflicts or payoff-definition differences stop the test. A large cell with zero payoffs also stops: do not interpret incomplete outcomes as actual zero.

Label-record presence alone does not prove every release/correction arrived. The next step preserves a completeness review. Never replace missing outcomes with zero or switch label sources silently.


In [9]:
q=follow.panel_audit_sql(PREP['panel']);dry(q,follow.READ_CAP)
AUDIT=follow.cached_query(CLIENT,'panel_audit',q,ROOT,follow.READ_CAP)
AUDIT.to_csv(ROOT/'panel_audit_raw.csv',index=False)
print(AUDIT.to_string(index=False))
INVENTORY=follow.validate_inventory(CENSUS,AUDIT)
INVENTORY.to_csv(ROOT/'verified_inventory_6_cells.csv',index=False)
coverage=INVENTORY[['feature_date','pool_coupon','member_rows','member_upb','payoff_events','missing_labels']].copy()
coverage['payoff_count_rate_pct']=100*coverage.payoff_events/coverage.member_rows
coverage.to_csv(ROOT/'monthly_outcome_coverage.csv',index=False)
display(coverage)


panel_audit job: 07a7184c-6c6d-4d6d-8953-0463f0acbaf9
feature_date  pool_coupon  panel_rows  distinct_panel_loans    panel_upb  payoff_events  missing_features  missing_labels  training_rule_ineligible_rows  label_definition_gaps  balance_conflicts
  2026-06-01          6.0      158754                158754 4.677318e+10           2097                 0               0                              0                      0                  0
  2026-06-01          6.5       83058                 83058 2.331798e+10           1627                 0               0                              0                      0                  0
  2026-07-01          6.0      156657                156657 4.605365e+10           1989                 0               0                              0                      0                  0
  2026-07-01          6.5       81431                 81431 2.278249e+10           1564                 0               0                              0              

,feature_date,pool_coupon,member_rows,member_upb,payoff_events,missing_labels,payoff_count_rate_pct
0,2026-06-01,6.0,158754,4.677318e+10,2097,0,1.320912
1,2026-06-01,6.5,83058,2.331798e+10,1627,0,1.958872
2,2026-07-01,6.0,156657,4.605365e+10,1989,0,1.269653
3,2026-07-01,6.5,81431,2.278249e+10,1564,0,1.920644
4,2026-08-01,6.0,154668,4.537949e+10,1649,0,1.066155
5,2026-08-01,6.5,79867,2.227030e+10,1249,0,1.563850


## Step 10 â€” Record outcome completeness and lock the experiment
Default June/July/August completeness is `unconfirmed`: you may score, but interpret results as provisional. Set `confirmed` only with actual evidence that relevant payoff terminations and servicing releases/corrections are complete. Nonzero events or table modification dates alone do not establish completeness.

You can leave the update dictionaries empty. Do not invent evidence. Record real batch IDs or reconciliation references if available. June has its own review because incomplete June outcomes also affect calibration.

This locks the same saved trees, encoder, matrix, 25 features, population and probability correction. No later outcomes may select calibration, features or stopping settings.


In [10]:
maturity_path=ROOT/'outcome_maturity_review.csv'
if not maturity_path.exists():
    pd.DataFrame([dict(month=mo,status='unconfirmed',evidence='') for mo in follow.EVAL_MONTHS]).to_csv(maturity_path,index=False)
MATURITY=pd.read_csv(maturity_path,keep_default_na=False)
MATURITY_UPDATES={}  # Optional: {month:{'status':'confirmed','evidence':'<actual completeness reference>'}}
for mo,d in MATURITY_UPDATES.items():
    assert mo in follow.EVAL_MONTHS and d['status'] in ['unconfirmed','provisional','confirmed']
    assert d['status']!='confirmed' or d.get('evidence','').strip()
    MATURITY.loc[MATURITY.month.eq(mo),['status','evidence']]=[d['status'],d.get('evidence','')]
MATURITY.to_csv(maturity_path,index=False)
june_review_path=ROOT/'june_outcome_review.json'
if not june_review_path.exists():core.write_json(june_review_path,dict(status='unconfirmed',evidence=''))
JUNE_OUTCOME_UPDATE=None  # Optional dict(status='confirmed',evidence='<actual completeness reference>')
if JUNE_OUTCOME_UPDATE is not None:core.write_json(june_review_path,JUNE_OUTCOME_UPDATE)
JUNE_REVIEW=json.loads(june_review_path.read_text())
SCANS=[]
for mo in follow.MONTHS:
    for role,q in zip(['input','predict'],follow.month_sql(PREP['panel'],mo)):
        (SQL/(role+'_'+mo+'.sql')).write_text(q)
        n=dry(q,follow.READ_CAP);SCANS.append(dict(month=mo,role=role,estimated_bytes=n,estimated_GiB=n/1024**3))
SCANS=pd.DataFrame(SCANS);SCANS.to_csv(ROOT/'scoring_estimates.csv',index=False)
assert EST.estimated_bytes.sum()+SCANS.estimated_bytes.sum()<=follow.TOTAL_CAP
SPEC=dict(version='freddie_saved_julaug_v1',months=follow.MONTHS,evaluation_months=follow.EVAL_MONTHS,
          panel=PREP['panel'],panel_identity=PANEL_ID,production_identity=PROD_ID,
          asset_paths=ASSETS,asset_sha256=OLD_SPEC['asset_sha256'],
          helper_sha256=sha(follow.__file__),core_sha256=sha(core.__file__),
          source_identities=PREP['source_identities'],roster_csv_sha256=PREP['roster_sha256'],
          original_run_spec_sha256=sha(INPUTS/'evaluation/run_spec.json'),
          training_start='2021-05-01',training_last_month='2026-05-01',
          sampling_odds_multiplier=.12,calibration_month='2026-06-01',prior_month_required=False,
          calibration_scope='one additional UPB-weighted June intercept per model, both G1 coupons combined',
          stages=follow.STAGES,primary_stage='before',models_retrained=False,
          production_calibration_replica=False,split_only_causal_test=False,independent_blind_test=False)
lock=ROOT/'run_spec.json'
if lock.exists():assert json.loads(lock.read_text())==SPEC,'Changed experiment: use a new run directory'
else:core.write_json(lock,SPEC)
print(SCANS.to_string(index=False));display(MATURITY);print('June review:',JUNE_REVIEW)


     month    role  estimated_bytes  estimated_GiB
2026-06-01   input         91921908       0.085609
2026-06-01 predict         81105422       0.075535
2026-07-01   input         90504906       0.084289
2026-07-01 predict         79880226       0.074394
2026-08-01   input         89148338       0.083026
2026-08-01 predict         78711289       0.073306


,month,status,evidence
0,2026-07-01,unconfirmed,
1,2026-08-01,unconfirmed,


June review: {'status': 'unconfirmed', 'evidence': ''}


## Step 11 â€” Score June and fit one additional factor per model
Both models already received the same 0.12 sampling correction. Solve a further odds multiplier so each model's June UPB-weighted average prediction matches June's UPB-weighted payoff rate.

Use **all matched June G1 loans, both coupons combined**, with the same weights in both arms. Do not fit separate factors by coupon, age, balance or delinquency. Freeze the two June factors before scoring July/August.

For probability `p` after sampling correction, the calibrated probability is `factor*p / (1-p+factor*p)`. A factor above 1 raises predictions; below 1 lowers them. The factor is additional to 0.12; the equivalent raw-odds factor is `0.12*factor`.


In [11]:
JUNE=follow.score_month(CLIENT,ROOT,SPEC,'2026-06-01',INVENTORY,EXPANDED,ENCODER,m)
assert (JUNE.actual_payoff.value_counts().reindex([0,1],fill_value=0)>=30).all(),'Too few June events/non-events'
FACTORS,CALIBRATION=follow.june_calibration(JUNE,m)
assert np.allclose(CALIBRATION.after_pct,CALIBRATION.actual_pct,atol=1e-6,rtol=0),'June factor did not match weighted actual rate'
FACTOR_LOCK=dict(run_spec_sha256=sha(lock),june_predictions_sha256=sha(ROOT/'months/2026-06-01/paired_predictions_before.parquet'),
                 factors=FACTORS,fit_month='2026-06-01',scope='overall matched Freddie G1, both coupons',
                 factor_scale='additional multiplier on already .12-corrected odds',production_calibration_replica=False)
factor_path=ROOT/'june_factors.json'
if factor_path.exists():assert json.loads(factor_path.read_text())==FACTOR_LOCK,'June factor changed: stop'
else:core.write_json(factor_path,FACTOR_LOCK)
CALIBRATION.to_csv(ROOT/'june_calibration_sensitivity.csv',index=False)
# Diagnostic within June, never counted as July/August performance.
june_reports=[]
for stage in follow.STAGES:
    x=JUNE.copy()
    if stage!='before':
        for arm in ['production','expanded']:
            factor=FACTORS[arm] if stage=='june_separate' else FACTORS['production']
            x[arm+'_before']=follow.odds(JUNE[arm+'_before'],factor)
    r=core.group_report(x,'2026-06-01');r['stage']=stage;r['role']='calibration';june_reports.append(r)
pd.concat(june_reports,ignore_index=True).to_csv(ROOT/'june_calibration_group_diagnostics.csv',index=False)
print(CALIBRATION.to_string(index=False))
del JUNE,june_reports,x;gc.collect()


input_2026-06-01 job: b2ba9df2-5ce5-4a25-a302-11b25eea0216
predict_2026-06-01 job: 4ea16d6f-afca-4ed6-95da-1e74bdee2093
     month  agency        arm                            scope  loans  payoff_events          upb  actual_pct  before_pct  after_pct  factor_on_before_probability  equivalent_factor_on_raw_probability
2026-06-01 freddie production all selected G1 coupons combined 241812           3724 7.009116e+10    1.598519    1.837274   1.598519                      0.867154                              0.104058
2026-06-01 freddie   expanded all selected G1 coupons combined 241812           3724 7.009116e+10    1.598519    1.746272   1.598519                      0.913220                              0.109586


37260

## Step 12 â€” Score July and August using the frozen trees and June factors
Within each month, all stages compare exactly the same loans and current-UPB weights. The populations can differ between months. Encoder categories are not refitted. Unknown categories are recorded for review.

A before-stage result is the main direct continuation of the historical Freddie comparison. June-calibrated results are a separate sensitivity. The common-factor stage helps show whether different June level adjustments contribute to a change in relative performance.


In [12]:
assert json.loads(factor_path.read_text())==FACTOR_LOCK
frames=[]
for mo in follow.EVAL_MONTHS:
    paired=follow.score_month(CLIENT,ROOT,SPEC,mo,INVENTORY,EXPANDED,ENCODER,m)
    frame=follow.report_stages(paired,mo,FACTORS)
    frame.to_csv(ROOT/'months'/mo/'accuracy_all_stages.csv',index=False);frames.append(frame)
    print(mo,'loans:',len(paired),'payoff events:',int(paired.actual_payoff.sum()))
    del paired;gc.collect()
RESULTS=pd.concat(frames,ignore_index=True)
MATURITY=pd.read_csv(maturity_path,keep_default_na=False)
RESULTS,PERIODS=follow.summarize(RESULTS,MATURITY,JUNE_REVIEW)
RESULTS.to_csv(ROOT/'all_evaluation_groups.csv',index=False)
PERIODS.to_csv(ROOT/'period_summary.csv',index=False)
NO_OFFSET=follow.coupon_no_offset_summary(RESULTS)
NO_OFFSET.to_csv(ROOT/'coupon_month_summary_without_offsetting.csv',index=False)
core.write_json(ROOT/'completion.json',dict(evaluation_months=follow.EVAL_MONTHS,calibration_month='2026-06-01',
 models_retrained=False,source_panel=PREP['panel'],primary_stage='before',independent_blind_test=False,
 all_evaluation_outcomes_confirmed=bool(MATURITY.status.eq('confirmed').all()),june_outcomes_confirmed=JUNE_REVIEW['status']=='confirmed'))


input_2026-07-01 job: ba6e8a63-ce91-41e2-a76e-2410ba200177
predict_2026-07-01 job: 8f4dc98d-3703-471d-a480-9dfae02189a3
2026-07-01 loans: 238088 payoff events: 3553
input_2026-08-01 job: 64b5aa58-a240-4bce-b8ed-f0f80d571007
predict_2026-08-01 job: 05771fcb-af99-4bbb-89b9-c7faf6d50e54
2026-08-01 loans: 234535 payoff events: 2898


## Step 13 â€” Review the main results and avoid cancellation
Positive gains favor the expanded model. Read all three metrics together:
- Absolute bias: how close the group's average prediction is to its actual payoff rate.
- Brier: probability error across individual loans.
- Log loss: probability accuracy, with a larger penalty for confident mistakes.

Rates and bias are monthly payoff percent and percentage points, not annual CPR. Period errors average monthly absolute errors using monthly UPB. UPB-month exposure repeats balances through time.

**Check both coupons separately.** An overprediction in 6.0 can cancel an underprediction in 6.5. `coupon_month_summary_without_offsetting.csv` takes the absolute error in each coupon-month cell before combining them, preventing that cancellation. No confidence intervals are calculated; small differences are not established as statistically significant.


In [13]:
cols=['month','stage','group','loans','payoff_events','actual_payoff_pct','production_predicted_pct','expanded_predicted_pct',
      'production_bias_pp','expanded_bias_pp','abs_bias_gain_pp','brier_gain','logloss_gain','outcome_status','calibration_outcome_status']
for name,key in [('monthly_overall','overall'),('monthly_coupon','coupon')]:
    frame=RESULTS.loc[RESULTS.grouping.eq(key)].sort_values(['stage','month','group'])
    frame.to_csv(ROOT/(name+'.csv'),index=False)
    print(name);display(frame[cols])
print('Period summary: overall and each coupon');display(PERIODS.loc[PERIODS.grouping.isin(['overall','coupon'])])
print('Coupon-month absolute errors without offsetting');display(NO_OFFSET)


monthly_overall


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status
0,2026-07-01,before,all,238088,3553,1.524889,1.669944,1.631897,0.145055,0.107008,0.038047,-2.518289e-06,0.000071,unconfirmed,not_applicable
537,2026-08-01,before,all,234535,2898,1.235998,1.633408,1.577649,0.397409,0.341651,0.055759,-1.549816e-05,0.000073,unconfirmed,not_applicable
358,2026-07-01,june_common_production,all,238088,3553,1.524889,1.452594,1.419908,-0.072295,-0.104981,-0.032686,5.153276e-07,0.000021,unconfirmed,unconfirmed
879,2026-08-01,june_common_production,all,234535,2898,1.235998,1.420730,1.372752,0.184732,0.136753,0.047978,-1.107812e-05,-0.000001,unconfirmed,unconfirmed
179,2026-07-01,june_separate,all,238088,3553,1.524889,1.452594,1.493575,-0.072295,-0.031314,0.040981,4.398694e-06,0.000056,unconfirmed,unconfirmed
708,2026-08-01,june_separate,all,234535,2898,1.235998,1.420730,1.443954,0.184732,0.207956,-0.023224,-1.235925e-05,-0.000090,unconfirmed,unconfirmed


monthly_coupon


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status
1,2026-07-01,before,6.0,156657,1989,1.256020,1.440167,1.381366,0.184147,0.125346,0.058801,1.889315e-05,0.000240,unconfirmed,not_applicable
81,2026-07-01,before,6.5,81431,1564,2.068393,2.134425,2.138331,0.066033,0.069938,-0.003906,-4.580043e-05,-0.000269,unconfirmed,not_applicable
538,2026-08-01,before,6.0,154668,1649,1.028722,1.437206,1.367169,0.408485,0.338447,0.070038,7.631814e-07,0.000232,unconfirmed,not_applicable
614,2026-08-01,before,6.5,79867,1249,1.658360,2.033201,2.006539,0.374842,0.348179,0.026663,-4.863340e-05,-0.000253,unconfirmed,not_applicable
359,2026-07-01,june_common_production,6.0,156657,1989,1.256020,1.252121,1.201510,-0.003900,-0.054510,-0.050610,1.997125e-05,0.000162,unconfirmed,unconfirmed
439,2026-07-01,june_common_production,6.5,81431,1564,2.068393,1.857841,1.861388,-0.210552,-0.207004,0.003547,-3.881384e-05,-0.000264,unconfirmed,unconfirmed
880,2026-08-01,june_common_production,6.0,154668,1649,1.028722,1.249609,1.189268,0.220888,0.160547,0.060341,3.373056e-06,0.000140,unconfirmed,unconfirmed
956,2026-08-01,june_common_production,6.5,79867,1249,1.658360,1.769417,1.746630,0.111057,0.088270,0.022787,-4.052483e-05,-0.000288,unconfirmed,unconfirmed
180,2026-07-01,june_separate,6.0,156657,1989,1.256020,1.252121,1.263995,-0.003900,0.007975,-0.004075,2.176578e-05,0.000174,unconfirmed,unconfirmed
260,2026-07-01,june_separate,6.5,81431,1564,2.068393,1.857841,1.957658,-0.210552,-0.110735,0.099817,-3.070801e-05,-0.000181,unconfirmed,unconfirmed


Period summary: overall and each coupon


,period,stage,grouping,group,months,expected_months,loan_months,upb_month_exposure_bn,all_outcomes_confirmed,calibration_outcomes_confirmed,complete_group_month_coverage,ready_for_final_interpretation,production_weighted_abs_bias_pp,production_weighted_signed_bias_pp,expanded_weighted_abs_bias_pp,expanded_weighted_signed_bias_pp,weighted_abs_bias_gain_pp,weighted_brier_gain,weighted_logloss_gain
22,jul_aug,before,coupon,6.0,2,2,311325,91.433136,False,True,True,False,0.295489,0.295489,0.231111,0.231111,0.064378,0.000010,0.000236
23,jul_aug,before,coupon,6.5,2,2,161298,45.052786,False,True,True,False,0.218682,0.218682,0.207477,0.207477,0.011205,-0.000047,-0.000261
38,jul_aug,before,overall,all,2,2,472623,136.485922,False,True,True,False,0.270135,0.270135,0.223309,0.223309,0.046826,-0.000009,0.000072
201,jul_aug,june_common_production,coupon,6.0,2,2,311325,91.433136,False,False,True,False,0.111594,0.107665,0.107137,0.052225,0.004456,0.000012,0.000151
202,jul_aug,june_common_production,coupon,6.5,2,2,161298,45.052786,False,False,True,False,0.161370,-0.051575,0.148312,-0.061045,0.013058,-0.000040,-0.000276
217,jul_aug,june_common_production,overall,all,2,2,472623,136.485922,False,False,True,False,0.128024,0.055101,0.120729,0.014836,0.007296,-0.000005,0.000010
380,jul_aug,june_separate,coupon,6.0,2,2,311325,91.433136,False,False,True,False,0.111594,0.107665,0.114375,0.114375,-0.002781,0.000012,0.000108
381,jul_aug,june_separate,coupon,6.5,2,2,161298,45.052786,False,False,True,False,0.161370,-0.051575,0.144289,0.032295,0.017081,-0.000036,-0.000268
396,jul_aug,june_separate,overall,all,2,2,472623,136.485922,False,False,True,False,0.128024,0.055101,0.118867,0.087281,0.009157,-0.000004,-0.000016


Coupon-month absolute errors without offsetting


,stage,coupon_month_cells,upb_month_exposure_bn,production_weighted_abs_bias_pp,expanded_weighted_abs_bias_pp,weighted_abs_bias_gain_pp,weighted_brier_gain,weighted_logloss_gain
0,before,4,136.485922,0.270135,0.223309,0.046826,-0.000009,0.000072
1,june_common_production,4,136.485922,0.128024,0.120729,0.007296,-0.000005,0.000010
2,june_separate,4,136.485922,0.128024,0.124249,0.003775,-0.000004,-0.000016


## Step 14 â€” Give delinquent and small loans a separate review
The historical Freddie experiment improved overall but worsened delinquent-loan errors. Review July and August separately for each coupon and calibration stage.

Small means original balance below $150,000, at all ages/statuses. Delinquent means `dlq_days>0`. Unknown status is shown separately. `status_balance` exposes delinquency Ã— original-balance groups. Age/balance tables include all ages. The earlier young-current focus is 12â€“23 months with incentive bands 0.5â€“1.0 for 6.0 and 1.0â€“1.5 for 6.5; as loans age, some groups may disappear. An absent group is not a zero-error group.

Check event counts and `small_group_attention`. Do not discard weak groups to improve a headline. Property type, state, occupancy and channel results are in the full CSV. These are same-loan model comparisons; group differences are not causal effects of balance or delinquency. The period summary records incomplete group-month coverage explicitly.


In [14]:
subsets={
 'monthly_small_loans':RESULTS.grouping.eq('small_all_ages'),
 'monthly_delinquent_loans':RESULTS.grouping.eq('status')&RESULTS['group'].str.endswith('_delinquent'),
 'monthly_delinquent_by_balance':RESULTS.grouping.eq('status_balance')&RESULTS['group'].str.contains('_delinquent_'),
 'monthly_age_original_balance':RESULTS.grouping.eq('age_original_balance'),
 'monthly_young_current_focus':RESULTS.grouping.eq('focus_current_young')}
for name,mask in subsets.items():
    g=RESULTS.loc[mask];g.to_csv(ROOT/(name+'.csv'),index=False)
    print(name);display(g.loc[g.stage.eq('before'),cols+['small_group_attention']])
priority=PERIODS.loc[PERIODS.grouping.eq('small_all_ages')|
 (PERIODS.grouping.eq('status')&PERIODS['group'].str.endswith('_delinquent'))|
 PERIODS.grouping.eq('focus_current_young')]
priority.to_csv(ROOT/'priority_group_period_summary.csv',index=False);display(priority)
category=pd.concat([pd.read_csv(ROOT/'months'/mo/'category_coverage.csv') for mo in follow.MONTHS],ignore_index=True)
category.to_csv(ROOT/'category_coverage_all_months.csv',index=False)
print('Unseen saved-encoder categories:');display(category.loc[category.unknown_rows.gt(0)])


monthly_small_loans


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status,small_group_attention
161,2026-07-01,before,6.0_lt150k,25301,365,1.369827,1.290645,1.238938,-0.079182,-0.130889,-0.051707,-0.000010,-0.000010,unconfirmed,not_applicable,False
170,2026-07-01,before,6.5_lt150k,16081,235,1.456207,1.566971,1.486430,0.110765,0.030223,0.080541,0.000025,0.000174,unconfirmed,not_applicable,False
690,2026-08-01,before,6.0_lt150k,24936,280,1.053471,1.264131,1.202838,0.210660,0.149367,0.061293,-0.000029,0.000113,unconfirmed,not_applicable,False
699,2026-08-01,before,6.5_lt150k,15846,215,1.258779,1.549294,1.442919,0.290515,0.184140,0.106375,-0.000023,0.000034,unconfirmed,not_applicable,False


monthly_delinquent_loans


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status,small_group_attention
15,2026-07-01,before,6.0_delinquent,3537,203,5.620535,4.187053,4.385364,-1.433481,-1.235171,0.198310,0.000687,0.002333,unconfirmed,not_applicable,False
95,2026-07-01,before,6.5_delinquent,2763,185,6.826519,4.922694,4.113150,-1.903825,-2.713370,-0.809544,-0.001065,-0.008155,unconfirmed,not_applicable,False
548,2026-08-01,before,6.0_delinquent,3404,168,5.372705,4.445643,4.732575,-0.927061,-0.640130,0.286931,-0.000126,0.000846,unconfirmed,not_applicable,False
624,2026-08-01,before,6.5_delinquent,2649,151,6.134826,4.702019,4.335661,-1.432808,-1.799165,-0.366357,-0.001248,-0.010000,unconfirmed,not_applicable,False


monthly_delinquent_by_balance


,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status,small_group_attention
163,2026-07-01,before,6.0_delinquent_lt150k,736,54,7.574880,4.268314,4.123907,-3.306566,-3.450973,-0.144407,-0.000518,-0.008939,unconfirmed,not_applicable,False
165,2026-07-01,before,6.0_delinquent_150_300k,1374,67,4.863104,4.292039,4.433969,-0.571066,-0.429135,0.141930,0.000035,-0.000641,unconfirmed,not_applicable,False
167,2026-07-01,before,6.0_delinquent_300_500k,905,53,5.793400,3.900346,4.102199,-1.893054,-1.691201,0.201853,0.000812,0.002866,unconfirmed,not_applicable,False
169,2026-07-01,before,6.0_delinquent_500kplus,522,29,5.661993,4.368738,4.696165,-1.293255,-0.965828,0.327427,0.001435,0.007163,unconfirmed,not_applicable,True
172,2026-07-01,before,6.5_delinquent_lt150k,588,33,5.510549,4.357815,3.635258,-1.152734,-1.875292,-0.722558,0.000667,0.002477,unconfirmed,not_applicable,False
174,2026-07-01,before,6.5_delinquent_150_300k,993,76,7.840924,4.787327,4.094400,-3.053597,-3.746524,-0.692927,-0.002932,-0.019562,unconfirmed,not_applicable,False
176,2026-07-01,before,6.5_delinquent_300_500k,683,39,5.930573,4.649135,3.878201,-1.281438,-2.052372,-0.770934,-0.001495,-0.010279,unconfirmed,not_applicable,False
178,2026-07-01,before,6.5_delinquent_500kplus,499,37,7.119543,5.343040,4.407138,-1.776503,-2.712405,-0.935902,0.000206,-0.000799,unconfirmed,not_applicable,False
692,2026-08-01,before,6.0_delinquent_lt150k,715,24,3.407952,4.111036,4.002130,0.703083,0.594177,0.108906,-0.001084,-0.002634,unconfirmed,not_applicable,True
694,2026-08-01,before,6.0_delinquent_150_300k,1288,71,5.644199,4.590256,4.822528,-1.053943,-0.821671,0.232272,-0.000269,0.000437,unconfirmed,not_applicable,False


... (3 outputs left out)

,month,stage,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,outcome_status,calibration_outcome_status,small_group_attention
2,2026-07-01,before,6.0_300_500k,5444,67,1.230900,1.488713,1.520330,0.257813,0.289431,-0.031617,-0.000004,0.000074,unconfirmed,not_applicable,False
3,2026-07-01,before,6.0_lt150k,2228,27,1.140996,1.247839,1.161888,0.106843,0.020892,0.085951,-0.000006,-0.000203,unconfirmed,not_applicable,True
4,2026-07-01,before,6.0_150_300k,5693,61,1.005448,1.310881,1.220854,0.305433,0.215406,0.090027,-0.000009,-0.000363,unconfirmed,not_applicable,False
5,2026-07-01,before,6.0_500kplus,2826,44,1.379382,1.898594,1.878268,0.519212,0.498886,0.020326,0.000041,0.001152,unconfirmed,not_applicable,False
82,2026-07-01,before,6.5_300_500k,2192,42,1.791979,2.423376,2.594424,0.631397,0.802445,-0.171048,-0.000024,0.000095,unconfirmed,not_applicable,False
83,2026-07-01,before,6.5_lt150k,1743,22,1.238942,1.522945,1.485809,0.284003,0.246867,0.037136,0.000004,0.000469,unconfirmed,not_applicable,True
84,2026-07-01,before,6.5_150_300k,2795,39,1.320696,1.915591,2.110014,0.594895,0.789318,-0.194423,-0.000014,0.000361,unconfirmed,not_applicable,False
85,2026-07-01,before,6.5_500kplus,1167,37,3.075670,2.835147,2.947829,-0.240523,-0.127841,0.112682,0.000055,0.001298,unconfirmed,not_applicable,False


,period,stage,grouping,group,months,expected_months,loan_months,upb_month_exposure_bn,all_outcomes_confirmed,calibration_outcomes_confirmed,complete_group_month_coverage,ready_for_final_interpretation,production_weighted_abs_bias_pp,production_weighted_signed_bias_pp,expanded_weighted_abs_bias_pp,expanded_weighted_signed_bias_pp,weighted_abs_bias_gain_pp,weighted_brier_gain,weighted_logloss_gain
24,jul_aug,before,focus_current_young,6.0_150_300k,1,2,5693,1.227312,False,True,False,False,0.305433,0.305433,0.215406,0.215406,0.090027,-9.328098e-06,-0.000363
25,jul_aug,before,focus_current_young,6.0_300_500k,1,2,5444,1.971610,False,True,False,False,0.257813,0.257813,0.289431,0.289431,-0.031617,-4.104460e-06,0.000074
26,jul_aug,before,focus_current_young,6.0_500kplus,1,2,2826,1.719707,False,True,False,False,0.519212,0.519212,0.498886,0.498886,0.020326,4.126741e-05,0.001152
27,jul_aug,before,focus_current_young,6.0_lt150k,1,2,2228,0.227813,False,True,False,False,0.106843,0.106843,0.020892,0.020892,0.085951,-6.337491e-06,-0.000203
28,jul_aug,before,focus_current_young,6.5_150_300k,1,2,2795,0.593000,False,True,False,False,0.594895,0.594895,0.789318,0.789318,-0.194423,-1.390128e-05,0.000361
29,jul_aug,before,focus_current_young,6.5_300_500k,1,2,2192,0.792887,False,True,False,False,0.631397,0.631397,0.802445,0.802445,-0.171048,-2.414148e-05,0.000095
30,jul_aug,before,focus_current_young,6.5_500kplus,1,2,1167,0.745112,False,True,False,False,0.240523,-0.240523,0.127841,-0.127841,0.112682,5.517277e-05,0.001298
31,jul_aug,before,focus_current_young,6.5_lt150k,1,2,1743,0.173679,False,True,False,False,0.284003,0.284003,0.246867,0.246867,0.037136,3.687583e-06,0.000469
49,jul_aug,before,small_all_ages,6.0_lt150k,2,2,50237,5.157734,False,True,True,False,0.144384,0.064554,0.140053,0.008094,0.004331,-1.965387e-05,0.000051
50,jul_aug,before,small_all_ages,6.5_lt150k,2,2,31927,3.193072,False,True,True,False,0.199888,0.199888,0.106538,0.106538,0.093350,1.146311e-06,0.000104


Unseen saved-encoder categories:


,month,feature,unknown_rows,unknown_upb_share


## Step 15 â€” Export both backups before closing Colab
The small ZIP contains code, specifications, audits and result CSVs. The FULL ZIP also contains the exact restored models, encoder, replay inputs, frozen loan-level June/July/August predictions and query caches. It is required for complete runtime recovery.

Files are created outside the run folder so the archive cannot include itself. Both ZIPs include SHA-256 manifests and are checked before download. Save both to `Y:/OASIVE/`. Do not close Colab until downloads complete. If the browser blocks the second automatic download, download that ZIP directly from Colab's Files sidebar. The Step 1 FULL-resume option uses this follow-up's own manifest.


In [15]:
from google.colab import files
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
PREFIX='g1_freddie_julaug_'+stamp
all_paths=[p for p in ROOT.rglob('*') if p.is_file() and '__pycache__' not in p.parts and not p.name.endswith('.recovering')]
small_paths=[p for p in all_paths if p.suffix.lower() in ['.csv','.json','.py','.sql','.md'] and 'inputs' not in p.relative_to(ROOT).parts]

def export_zip(path,paths,manifest_name):
    hashes={p.relative_to(ROOT).as_posix():sha(p) for p in paths}
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:
        for p in paths:z.write(p,p.relative_to(ROOT).as_posix())
        z.writestr(manifest_name,json.dumps(hashes,indent=2))
    with zipfile.ZipFile(path) as z:
        assert z.testzip() is None,'Incomplete ZIP'
        for n,h in hashes.items():
            digest=hashlib.sha256()
            with z.open(n) as f:
                for b in iter(lambda:f.read(1024**2),b''):digest.update(b)
            assert digest.hexdigest()==h,'Backup checksum mismatch: '+n
    print(path,'bytes:',path.stat().st_size)
SMALL=Path('/content')/(PREFIX+'_small_results.zip')
FULL=Path('/content')/(PREFIX+'_FULL_recovery.zip')
export_zip(SMALL,small_paths,'freddie_julaug_small_hashes.json')
export_zip(FULL,all_paths,'freddie_julaug_recovery_hashes.json')
files.download(str(SMALL))
files.download(str(FULL))


/content/g1_freddie_julaug_20261102T234612_small_results.zip bytes: 514712
/content/g1_freddie_julaug_20261102T234612_FULL_recovery.zip bytes: 79432236


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## How to decide what the results mean
1. Start with July and August separately, `before`, overall and by coupon. Look for improvement in absolute bias, Brier and log loss together. Do not infer a broad win from only one metric.
2. Inspect delinquent loans first, then small loans and age/balance groups. A historical March win does not excuse later regressions.
3. Review `june_separate` separately. If calibration reverses the comparison, report that the overall June level adjustment contributes to the result. Check June diagnostics and non-offsetting coupon errors.
4. A July gain followed by an August loss is mixed evidence. The same applies to one coupon improving while the other worsens. State magnitudes, exposure and events.
5. Treat unconfirmed outcomes and calibration as provisional. This snapshot evaluation does not certify point-in-time feature availability. Do not claim September results: September is outside this notebook.
6. Stable later-month improvements would support a subsequent paired-seed robustness experiment. These results alone cannot identify the validation split as the cause, because production and the local model also differ in fitting engine, sampling/agency scope and stopping settings.

## Files produced and where to find them
Local Colab root: `/content/g1_freddie_julaug_20261102/`.

| File | Contents |
|---|---|
| `saved_model_replay.json` | March replay check |
| `preparation_spec.json`, `run_spec.json` | Frozen sources, model hashes, dates and exact sandbox table IDs |
| `membership_census.csv`, `panel_audit_raw.csv`, `verified_inventory_6_cells.csv` | Matched population and join/label/balance checks |
| `monthly_outcome_coverage.csv`, `outcome_maturity_review.csv`, `june_outcome_review.json` | Outcome coverage and evidence status |
| `june_calibration_sensitivity.csv`, `june_factors.json` | Two June-only factors, scope and probability scale |
| `june_calibration_group_diagnostics.csv` | June diagnostics; excluded from evaluation summaries |
| `months/YYYY-MM-01/paired_predictions_before.parquet` | Frozen loan-level predictions before June adjustments |
| `months/YYYY-MM-01/accuracy_all_stages.csv` | July/August group comparisons across stages |
| `monthly_overall.csv`, `monthly_coupon.csv` | Main monthly comparisons |
| `period_summary.csv`, `coupon_month_summary_without_offsetting.csv` | Two-month summaries with monthly and coupon cancellation checks |
| `monthly_small_loans.csv`, `monthly_delinquent_loans.csv`, `monthly_delinquent_by_balance.csv` | Priority risk groups |
| `monthly_age_original_balance.csv`, `monthly_young_current_focus.csv` | Age/balance detail and previous focus groups |
| `priority_group_period_summary.csv`, `all_evaluation_groups.csv` | Full group results |
| `category_coverage_all_months.csv` | Unknown saved-encoder categories |
| `code/g1_freddie_julaug.py` | This notebook's scoring/calibration helper |
| `inputs/tools/g1_saved_payoff_comparison_core.py` | Exact archived shared scoring code |
| `inputs/original/original_saved_experiment/post_rally_matched_test.py` | Exact original feature-matrix and calibration solver module |
| `sql/`, `query_cache/` | Exact SQL, job IDs and resumable cached results |

The setup builder, notebook, standalone helper and this Markdown guide are saved together under `Y:/OASIVE/oasive-quant/docs/research/g1_refinance_plan_20261022/freddie_julaug_20261102/`. The code cells above contain the complete implementation. Development validation passed 18 focused tests, checked all 13 recovery members against the supplied archive manifest, and dry-ran the census, panel freeze and virtual August production prediction. Actual model loading/replay and the new payoff comparisons run in your Colab session; no new results have been produced here. This guide does not contain results from the new July/August run; those are produced when you execute it.
